# GCI 2026 — RealMLP and TabM shared-fold experiment

This notebook runs the official PyTabKit RealMLP-TD classifier and official TabM architecture on the supplied competition data. It reuses the packaged feature builder, fixed three-fold assignments, and LightGBM OOF control. No champion artifact is modified.

**Runtime:** select a Colab GPU if available (Runtime → Change runtime type). The notebook checks the actual device and runs on CPU if CUDA is unavailable. Expect model runs to take longer on CPU. It first fits both real architectures on a stratified 3,000-row smoke sample, then proceeds with all three full-data folds for each candidate that passes its smoke test.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Upload exactly one gci_realmlp_tabm_colab_bundle.zip; found {zip_names}')

WORK = Path('/content/gci_realmlp_tabm')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    for member in archive.infolist():
        target = (WORK / member.filename).resolve()
        if WORK.resolve() not in target.parents and target != WORK.resolve():
            raise ValueError(f'Unsafe ZIP path: {member.filename}')
    archive.extractall(WORK)

PROJECT = WORK
DATA = PROJECT / 'Competition' / 'input'
RESEARCH = PROJECT / 'research' / 'realmlp_tabm'
for required in [DATA / 'train.csv', DATA / 'test.csv', DATA / 'sample_submission.csv',
                 PROJECT / 'feats.py', RESEARCH / 'run_experiments.py',
                 RESEARCH / 'fold_assignments.csv', RESEARCH / 'oof_tree.csv',
                 RESEARCH / 'fold_metrics.csv']:
    if not required.is_file():
        raise FileNotFoundError(f'Required bundle file is missing: {required}')
print('Extracted package:', PROJECT)
print('Input CSV sizes (MiB):', {p.name: round(p.stat().st_size / 2**20, 2) for p in DATA.glob('*.csv')})

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

print('Python:', sys.version.replace('\n', ' '))
try:
    import torch
except Exception as exc:
    raise RuntimeError('Colab could not import its provided PyTorch. No PyTorch reinstall or numerical-stack upgrade was attempted.') from exc

if importlib.util.find_spec('pytabkit') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', 'pytabkit==1.7.3'])
if importlib.util.find_spec('tabm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', 'tabm==0.0.3'])

# Install only missing direct runtime dependencies; never ask pip to resolve or replace torch/NumPy/pandas/SciPy/sklearn.
runtime_packages = {
    'pytorch_lightning': 'pytorch-lightning',
    'torchmetrics': 'torchmetrics',
    'psutil': 'psutil',
    'rtdl_num_embeddings': 'rtdl_num_embeddings==0.0.12',
    'typing_extensions': 'typing_extensions',
    'lightning_utilities': 'lightning-utilities',
    'yaml': 'PyYAML', 'tqdm': 'tqdm', 'packaging': 'packaging',
    'fsspec': 'fsspec', 'filelock': 'filelock',
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn'
}
missing = [package for module, package in runtime_packages.items()
           if importlib.util.find_spec(module) is None and module not in {'numpy', 'pandas', 'scipy', 'sklearn'}]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', *missing])
for module in ['numpy', 'pandas', 'scipy', 'sklearn']:
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f'Colab is missing required data-science package {module!r}; it was not upgraded automatically.')

import numpy, pandas, scipy, sklearn
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('Device:', DEVICE, '| device name:', DEVICE_NAME, '| CUDA available:', torch.cuda.is_available())
ENVIRONMENT = {'python': sys.version.split()[0], 'device': DEVICE, 'device_name': DEVICE_NAME,
               'cuda_available': bool(torch.cuda.is_available())}
for package in ['torch', 'pytabkit', 'tabm', 'numpy', 'pandas', 'scipy', 'scikit-learn', 'pytorch-lightning', 'torchmetrics']:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = 'not installed'
    ENVIRONMENT[package] = version
    print(f'{package}: {version}')

In [ ]:
import importlib
import inspect
import traceback
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(RESEARCH))
from run_experiments import (CATS, N_FOLDS, SEED, read_data, make_folds,
                             realmlp_fit_predict, tabm_fit_predict)

api_status = {}
for name, module_name, class_name in [('realmlp', 'pytabkit', 'RealMLP_TD_Classifier'),
                                      ('tabm', 'tabm', 'TabM')]:
    try:
        module = importlib.import_module(module_name)
        model_class = getattr(module, class_name)
        signature = inspect.signature(model_class)
        if name == 'realmlp':
            assert 'n_epochs' in signature.parameters and 'device' in signature.parameters
            print('Official PyTabKit entry point:', model_class.__module__)
            print('RealMLP supported parameters:', signature)
        else:
            make_signature = inspect.signature(model_class.make)
            assert 'd_out' in signature.parameters and 'cat_cardinalities' in signature.parameters
            print('Official TabM package:', metadata.version('tabm'))
            print('TabM constructor:', signature, '| TabM.make:', make_signature)
        api_status[name] = 'verified'
    except Exception:
        api_status[name] = traceback.format_exc()
        print(name, 'API verification failed; this candidate will be skipped:')
        print(api_status[name])
print('CUDA runtime verified:', torch.cuda.is_available(), '| selected device:', DEVICE)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

X, y, ids, raw_categories = read_data()
folds = make_folds(X, y, ids)
train_csv = pd.read_csv(DATA / 'train.csv', usecols=['SK_ID_CURR', 'TARGET'])
test_ids = pd.read_csv(DATA / 'test.csv', usecols=['SK_ID_CURR'])['SK_ID_CURR'].to_numpy()
sample = pd.read_csv(DATA / 'sample_submission.csv')
fold_map = pd.read_csv(RESEARCH / 'fold_assignments.csv')
tree = pd.read_csv(RESEARCH / 'oof_tree.csv')
assert len(X) == len(y) == 171202
assert len(test_ids) == len(sample) == 61500 and np.array_equal(test_ids, sample.SK_ID_CURR.to_numpy())
assert np.array_equal(ids, train_csv.SK_ID_CURR.to_numpy())
assert np.array_equal(y, train_csv.TARGET.to_numpy())
assert np.array_equal(fold_map.SK_ID_CURR.to_numpy(), ids)
assert np.array_equal(fold_map.TARGET.to_numpy(), y)
assert tree.shape[0] == len(y)
assert np.array_equal(tree.SK_ID_CURR.to_numpy(), ids) and np.array_equal(tree.TARGET.to_numpy(), y)
assert np.array_equal(tree.fold.to_numpy(), fold_map.fold.to_numpy())
assert np.isfinite(tree.prediction.to_numpy()).all()
reproduced = np.full(len(y), -1, dtype=np.int8)
for fold_no, (_, valid_idx) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y), 1):
    reproduced[valid_idx] = fold_no
assert np.array_equal(reproduced, fold_map.fold.to_numpy())
assert set(fold_map.fold.unique()) == {1, 2, 3}
print('Train/test rows:', len(y), len(test_ids), '| target prevalence:', round(float(y.mean()), 6))
print('Features:', X.shape[1], '| fold counts:', fold_map.fold.value_counts().sort_index().to_dict())
print('Tree baseline pooled OOF AUC:', round(roc_auc_score(y, tree.prediction), 6))
print('All input, target, identifier, fold and baseline OOF alignment checks passed.')

## Genuine architecture smoke tests

Each official model is fitted on a stratified 3,000-row subset derived from the competition data. This is only a runtime/API/prediction-shape check; its AUC is not treated as a competitive result. A failure is logged and does not prevent the other architecture from being tried.

In [ ]:
import os, traceback, time
from sklearn.model_selection import train_test_split

smoke_idx, _ = train_test_split(np.arange(len(y)), train_size=3000, random_state=2026,
                                stratify=y)
smoke_train, smoke_valid = train_test_split(smoke_idx, test_size=0.25, random_state=2027,
                                           stratify=y[smoke_idx])
smoke_X = X.drop(columns=CATS).copy()
for col in CATS:
    smoke_X[col] = raw_categories[col].to_numpy()
smoke_results = {}
for name, fit_predict, epoch_name, smoke_epochs in [
    ('realmlp', realmlp_fit_predict, 'REALMLP_EPOCHS', 4),
    ('tabm', tabm_fit_predict, 'TABM_EPOCHS', 2),
]:
    os.environ['NEURAL_DEVICE'] = DEVICE
    os.environ[epoch_name] = str(smoke_epochs)
    try:
        if api_status.get(name) != 'verified':
            raise RuntimeError('Official package API was not verified; see API verification output.')
        prediction, seconds, _ = fit_predict(smoke_X, y, smoke_train, smoke_valid, 1)
        assert prediction.shape == (len(smoke_valid),) and np.isfinite(prediction).all()
        assert ((prediction >= 0) & (prediction <= 1)).all()
        score = float(roc_auc_score(y[smoke_valid], prediction))
        smoke_results[name] = {'status': 'passed', 'auc_smoke_only': score, 'seconds': seconds}
        print(name, smoke_results[name])
    except Exception:
        smoke_results[name] = {'status': 'failed', 'traceback': traceback.format_exc()}
        print(name, 'smoke failed; will skip this candidate and continue:')
        print(smoke_results[name]['traceback'])
    finally:
        os.environ.pop(epoch_name, None)

if not any(result['status'] == 'passed' for result in smoke_results.values()):
    print('No candidate passed smoke; full runs will be skipped and the failure report will still be exported.')

## Full shared-fold runs

The runner reuses the exact saved three folds and feature pipeline. RealMLP and TabM use fold-local preprocessing. A smoke-test failure skips only that model; a full-run failure is captured and the next candidate is attempted. No tree retraining is performed.

In [ ]:
import subprocess

run_logs = {}
for name, smoke_key, epoch_name, full_epochs in [
    ('realmlp', 'realmlp', 'REALMLP_EPOCHS', 64),
    ('tabm', 'tabm', 'TABM_EPOCHS', 20),
]:
    if smoke_results[name]['status'] != 'passed':
        run_logs[name] = 'Skipped because its real-model smoke test failed.'
        continue
    env = os.environ.copy()
    env['GCI_INPUT'] = str(DATA)
    env['NEURAL_DEVICE'] = DEVICE
    env[epoch_name] = str(full_epochs)
    command = [sys.executable, str(RESEARCH / 'run_experiments.py'), '--model', name]
    process = subprocess.Popen(command, cwd=PROJECT, env=env, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               bufsize=1)
    output_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        output_lines.append(line)
    return_code = process.wait()
    output = ''.join(output_lines)
    log_path = RESEARCH / f'{name}_colab.log'
    log_path.write_text(output, encoding='utf-8')
    run_logs[name] = output
    if return_code:
        print(f'{name} full shared-fold run failed with exit code {return_code}; continuing.')
    else:
        candidate_oof = RESEARCH / f'oof_{name}.csv'
        if not candidate_oof.is_file():
            raise FileNotFoundError(f'Runner reported success but did not produce {candidate_oof}')

print('Completed candidates:', [name for name in ['realmlp', 'tabm']
                                if (RESEARCH / f'oof_{name}.csv').is_file()])

In [ ]:
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import log_loss

metrics_path = RESEARCH / 'fold_metrics.csv'
all_metrics = pd.read_csv(metrics_path)
baseline = pd.read_csv(RESEARCH / 'oof_tree.csv')
blend_rows = []
candidate_summary = []
blend_weights = [0.05, 0.10, 0.20]
baseline_rank = (rankdata(baseline.prediction.to_numpy()) - 1) / (len(y) - 1)
baseline_auc = float(roc_auc_score(y, baseline.prediction))

for name in ['realmlp', 'tabm']:
    path = RESEARCH / f'oof_{name}.csv'
    if not path.is_file():
        candidate_summary.append({'model': name, 'verdict': 'INVESTIGATE FURTHER',
                                  'status': 'No complete full-fold OOF file; see captured log.'})
        continue
    pred = pd.read_csv(path)
    assert len(pred) == len(y)
    assert np.array_equal(pred.SK_ID_CURR.to_numpy(), ids)
    assert np.array_equal(pred.TARGET.to_numpy(), y)
    assert np.array_equal(pred.fold.to_numpy(), fold_map.fold.to_numpy())
    assert np.isfinite(pred.prediction.to_numpy()).all()
    candidate_rank = (rankdata(pred.prediction.to_numpy()) - 1) / (len(y) - 1)
    corr = float(spearmanr(baseline.prediction, pred.prediction).statistic)
    scores = [roc_auc_score(y[fold_map.fold.to_numpy() == f], pred.prediction.to_numpy()[fold_map.fold.to_numpy() == f])
              for f in [1, 2, 3]]
    model_metrics = all_metrics[(all_metrics.model == ('pytabkit_realmlp_td' if name == 'realmlp' else 'official_tabm'))]
    candidate_summary.append({
        'model': name,
        'verdict': 'PENDING BLEND CHECK',
        'fold_auc': [round(float(s), 6) for s in scores],
        'mean_fold_auc': float(np.mean(scores)),
        'sd_fold_auc': float(np.std(scores, ddof=1)),
        'pooled_oof_auc': float(roc_auc_score(y, pred.prediction)),
        'pooled_log_loss': float(log_loss(y, np.clip(pred.prediction, 1e-7, 1 - 1e-7))),
        'tree_spearman': corr,
        'fit_seconds_total': float(model_metrics.fit_runtime_sec.sum()),
        'device': DEVICE,
    })
    for weight in blend_weights:
        blend = (1 - weight) * baseline_rank + weight * candidate_rank
        fold_auc = [roc_auc_score(y[fold_map.fold.to_numpy() == f], blend[fold_map.fold.to_numpy() == f])
                    for f in [1, 2, 3]]
        base_fold_auc = [roc_auc_score(y[fold_map.fold.to_numpy() == f], baseline.prediction.to_numpy()[fold_map.fold.to_numpy() == f]) for f in [1, 2, 3]]
        blend_rows.append({'model': name, 'neural_weight': weight,
                           'pooled_oof_auc': float(roc_auc_score(y, blend)),
                           'fold_auc': [float(score) for score in fold_auc],
                           'fold_auc_deltas_vs_tree': [float(fold_auc[i] - base_fold_auc[i]) for i in range(3)]})

for summary in candidate_summary:
    if summary.get('pooled_oof_auc') is None:
        continue
    candidate_fold = [row for row in blend_rows if row['model'] == summary['model']]
    standalone_win_count = sum(score > base for score, base in zip(summary['fold_auc'],
        [roc_auc_score(y[fold_map.fold.to_numpy() == f], baseline.prediction.to_numpy()[fold_map.fold.to_numpy() == f]) for f in [1, 2, 3]]))
    blend_has_stable_gain = any(row['pooled_oof_auc'] > baseline_auc and
                                sum(delta > 0 for delta in row['fold_auc_deltas_vs_tree']) >= 2
                                for row in candidate_fold)
    if summary['pooled_oof_auc'] >= baseline_auc and standalone_win_count >= 2:
        summary['verdict'] = 'KEEP'
    elif blend_has_stable_gain:
        summary['verdict'] = 'INVESTIGATE FURTHER'
    else:
        summary['verdict'] = 'REJECT'

pd.DataFrame(blend_rows).to_csv(RESEARCH / 'blend_metrics.csv', index=False)
pd.DataFrame(candidate_summary).to_json(RESEARCH / 'candidate_summary.json', orient='records', indent=2)
print('Candidate summary:')
print(pd.DataFrame(candidate_summary).to_string(index=False))
print('\nPredefined OOF blend checks (weight-selection optimism applies):')
print(pd.DataFrame(blend_rows).to_string(index=False))
print('\nNote: even a favorable blend here is only against the single LightGBM control, not the historical champion.')

In [ ]:
import json, zipfile

report = [
    '# Colab RealMLP and TabM experiment results', '',
    f'- Environment: `{json.dumps(ENVIRONMENT)}`',
    f'- Data rows: {len(y)}; fixed folds: {N_FOLDS}; seed: {SEED}',
    f'- Tree control pooled OOF AUC: {roc_auc_score(y, baseline.prediction):.6f}', '',
    '## Smoke tests', '', '```json', json.dumps(smoke_results, indent=2), '```', '',
    '## Candidates', '', '```json', json.dumps(candidate_summary, indent=2), '```', '',
    '## Predefined rank blends', '', '```json', json.dumps(blend_rows, indent=2), '```', '',
    'Blend weights were predefined (5%, 10%, 20%) and evaluated on these OOF predictions; this introduces model-selection optimism.',
    'A blend is compared only with the single LightGBM control and cannot establish improvement over the historical champion.'
]
(RESEARCH / 'colab_experiment_report.md').write_text('\n'.join(report) + '\n', encoding='utf-8')

OUTPUT_ZIP = Path('/content/gci_realmlp_tabm_colab_results.zip')
allowed = [metrics_path, RESEARCH / 'fold_assignments.csv', RESEARCH / 'oof_tree.csv',
           RESEARCH / 'blend_metrics.csv', RESEARCH / 'candidate_summary.json',
           RESEARCH / 'colab_experiment_report.md']
for name in ['realmlp', 'tabm']:
    allowed.extend([RESEARCH / f'oof_{name}.csv', RESEARCH / f'{name}_colab.log'])
with zipfile.ZipFile(OUTPUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in allowed:
        if path.is_file():
            archive.write(path, arcname=path.name)
print('Results archive:', OUTPUT_ZIP, '| size MiB:', round(OUTPUT_ZIP.stat().st_size / 2**20, 2))
files.download(str(OUTPUT_ZIP))